In [1]:
# Make sure GPU is enabled
import torch
device = 0 if torch.cuda.is_available() else -1

if device == 0:
    print("Device found")
else:
    print("No device")

Device found


In [2]:
import pandas as pd
import torch
from sklearn.preprocessing import MultiLabelBinarizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline
from torch.utils.data import Dataset

# Step 1: Load dataset
df = pd.read_csv('mlc_dataset.csv')

/home/cpsc415_jk2783/.conda/envs/elevator_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
print(f"Data: {df.shape}")
df.head(5)

Data: (23, 2)


,Essay,Labels
0,Winnie-the-Pooh would've shuddered seeing hone...,"[""Resume Dumping"", ""Eureka Problem"", ""Needs Mo..."
1,Libraries taught me resilience: a place where ...,"[""Weak Reflection"", ""Needs More Specificity""]"
2,Folding paper into origami shapes was how I di...,"[""Eureka Problem"", ""Needs Conflict Arc"", ""Mino..."
3,"Walking into the coffee club meeting, I felt l...","[""Needs More Specificity"", ""Tone Awareness""]"
4,"As I stared at my grandmother’s hands, the mem...","[""Weak Conclusion"", ""Needs More VSPICE"", ""Weak..."


In [4]:
# Step 2: Preprocess labels
df['Labels'] = df['Labels'].apply(eval)  # Turn string list into actual list
mlb = MultiLabelBinarizer()
label_matrix = mlb.fit_transform(df['Labels'])
label_names = mlb.classes_.tolist()

In [5]:
# Example:

print(f"Essay: {df['Essay'][0][:100]}")
print(f"Labels: {df['Labels'][0]}")
print(f"Label encoding: {label_matrix[0]}")

Essay: Winnie-the-Pooh would've shuddered seeing honey, a colony’s lifeblood, bleeding from one of nature’s
Labels: ['Resume Dumping', 'Eureka Problem', 'Needs More Specificity', 'Weak Conclusion', 'Needs More Perseverance']
Label encoding: [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 1 0 0 0 0
 0 0 0 0 0 0 0 1 0 0 0 0 0]


In [6]:
label_names

['Abstract Over Tangible Moments',
 'Could Use Clearer Action Arc',
 'Covers Too Much Content',
 'Eureka Problem',
 'Expand Conflict Details',
 'Expand Initial Failure Scene',
 'Expand Personal Reflections',
 'Flow Improvement',
 'Flow Slightly Disjointed',
 'Minor Flow Adjustments',
 'Minor Reflection Weakness',
 'Minor Timeline Clarification',
 'Minor Word Choice Fixes',
 'Narrative Tightening Needed',
 'Needs Clearer Central Theme',
 'Needs Clearer Early Stakes',
 'Needs Clearer Stakes',
 'Needs Clearer Timeline',
 'Needs Conflict Arc',
 'Needs Deeper Early Conflict',
 'Needs Deeper Storytelling',
 'Needs More Concrete Conflict',
 'Needs More Perseverance',
 'Needs More Self-focus',
 'Needs More Specific Scenes',
 'Needs More Specificity',
 'Needs More Structure',
 'Needs More Tangible Examples',
 'Needs More VSPICE',
 'Needs Stronger Conclusion',
 'Needs Stronger Transitions',
 'Reflection Expansion Needed',
 'Resume Dumping',
 'Slight Overgeneralization',
 'Smoother Transition to 

In [15]:
# Step 3: Tokenization
from transformers import DebertaV2Tokenizer
MODEL_NAME = "microsoft/deberta-v3-base"  # Use DeBERTa
tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)


class EssayDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=512)
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

In [16]:
# Step 4: Create dataset
dataset = EssayDataset(df['Essay'].tolist(), label_matrix, tokenizer)

In [20]:
# Step 5: Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(label_names),
    problem_type="multi_label_classification"
)

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
# Step 6: Training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=8,
    per_device_train_batch_size=4,
    learning_rate=2e-5,
    eval_strategy="no",  # Small dataset, skip eval for now
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=10,
)

In [23]:
# Step 7: Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [24]:
# Step 8: Fine-tune!
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: toddcobble (toddcobble-yale-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/home/cpsc415_jk2783/.conda/envs/elevator_env/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
10,0.674800
20,0.632400


/home/cpsc415_jk2783/.conda/envs/elevator_env/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/cpsc415_jk2783/.conda/envs/elevator_env/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/cpsc415_jk2783/.conda/envs/elevator_env/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/cpsc415_jk2783/.conda/envs/elevator_env/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze an

TrainOutput(global_step=24, training_loss=0.6463595529397329, metrics={'train_runtime': 34.72, 'train_samples_per_second': 5.3, 'train_steps_per_second': 0.691, 'total_flos': 48434166841344.0, 'train_loss': 0.6463595529397329, 'epoch': 8.0})

In [27]:
# Step 9: Save the model + tokenizer
import json
model.save_pretrained("./deberta_mlc_model")
tokenizer.save_pretrained("./deberta_mlc_model")

# Also save label names
with open("./deberta_mlc_model/labels.json", "w") as f:
    json.dump(label_names, f)

print("Model, tokenizer, and labels saved!")

Model, tokenizer, and labels saved!


In [28]:
# ------------------
# Step 10: Load pipeline for inference on new essays
# ------------------

def load_pipeline():
    model = AutoModelForSequenceClassification.from_pretrained("./deberta_mlc_model")
    tokenizer = DebertaV2Tokenizer.from_pretrained("./deberta_mlc_model")
    with open("./deberta_mlc_model/labels.json", "r") as f:
        labels = json.load(f)

    clf_pipeline = pipeline("text-classification", model=model, tokenizer=tokenizer, return_all_scores=True)
    return clf_pipeline, labels

In [31]:

# Example usage:

clf_pipeline, label_names = load_pipeline()
sample_essay = """
Winnie-the-Pooh would've shuddered seeing honey, a colony’s lifeblood, bleeding from one of nature’s miracles—the once buzzing beehive. My beekeeping suit offered no protection from the massacre: thousands of contorted, lifeless bees victims of a vicious pest larvae army. A haunting odor—the combination of wet socks and decomposing oranges—clung to my nostrils. 

The root of this destruction? A small hive beetle. 

Over the past 6 years, I’ve collaborated with Dr. Charles Stuhl at the USDA, independently prototyping treatments/devices and running experiments with commercial beekeepers. More specifically, I've been researching the global issue of eliminating invasive small hive beetles in an eco-friendly, low-cost manner.  To me, research is a superpower, my tool to improve ecology and benefit farmers, the cornerstones of our food supply. Wielding my superpower to address real-world problems, I create “research-for-impact.”

When I was twelve  I met with a friend and his grandfather, a generational beekeeper, who told me how these pest infestations had destroyed his apiaries. In addition to being drawn to these beautiful black and yellow creatures, I wanted to help solve his problem. So, I devoured  every bee-related book I found, I learned that one-in-three bites of food comes from bees and 50% of US hives are dying annually. Afterwards, I attended Dr. Jamie Ellis' honeybee anatomy lecture at the University of Florida's Bee College Conference. Later working in his lab, I researched commercial pollen substitutes to support hive growth. My findings, published 1st author in a peer-reviewed journal, fueled my passion to aid beekeepers; I was hooked.

However, it was the destructive small hive beetle that truly captured my attention. At the time, beekeepers had two treatment options: expensive/toxic pesticides or ineffective organics.  

Were those truly our only options? As someone who was used to upcycling plastic bottles as plant vases, surely I could find a do-it-yourself option around my house… right? My first mission was determining the most attractive natural bait—oils, yeasts, fruit purees, beer,—placed in plastic in-hive traps. The beetles, like humans, were lured by an irresistible organic bait: beer. Believe it or not, Bud Light  proved highly-effective, outperforming the leading organic option by 33-times!


The following year, I harnessed the power of machine learning and IoT technology to create BeetleGuardAI–the world’s first 3D-printed solar-powered beetle trap. The downside of my work? Getting hospitalized for three-dozen stings. The upside? Enough honey for Pooh, Piglet and Tigger. 

Undoubtedly, the greatest gift of research-for-impact is sharing my ideas. 

Witnessing my love of beekeeping impacted by climate change leads me to represent youth at UN conferences, lead STEM workshops in schools, read children’s books in libraries, and cal for action for sustainable agriculture. 

The gratitude in beekeepers’ eyes, the sight of thriving colonies, and the knowledge that I'm making a difference–these are the moments that energize me. However, nothing compares to lifting a hive's lid, the sweet aroma of honey and harmonious buzzing symphony relaxing me. Most people would recoil seeing hundreds of dead beetles drowned in my beer traps. In my eyes, the annihilation of these colony-killers represents a victory, not just for beekeepers and farmers, but for “research-for-impact”.  Witnessing our ecosystem’s fragility instills me with profound environmental responsibility and commitment to harnessing sustainable innovation to protect what matters most. 

Through my experiences, I’ve learned to emphasize curiosity and unwavering determination. Driven by my love of learning and desire to positively impact society, I’m dedicated to further amplifying my impact. After all, a world without honey is a world without Pooh-Bear, and that’s just un-bearable.

"""
outputs = clf_pipeline(sample_essay)
threshold = 0.5
predicted_labels = [label_names[i] for i, score in enumerate(outputs[0]) if score['score'] > threshold]
print(predicted_labels)

Device set to use cuda:0


['Expand Initial Failure Scene', 'Needs Conflict Arc', 'Needs More Concrete Conflict', 'Smoother Transition to Broader Reflection', 'Strengthen Closing Reflection', 'Too Abstract', 'Weaker Reflection Arc']


In [34]:

# Example usage:

clf_pipeline, label_names = load_pipeline()
new_essay = """
The world is not your canvas. A canvas doesn’t fold or pleat, nor crumple and crease; it can’t shape cranes that flap their wings, nor can it conceal a six-year old’s crayon doodle on the wall. I prefer to think of life as a sheet of paper— which is undoubtedly similar, yet allows for another dimension of creativity that is lost in a rigid canvas. And although a stack of this everyday object is just a $17.99 purchase from the Staples next door, it remains irreplaceable in my heart.
I was no art prodigy; in fact, I wasn’t even that talented. I can easily list a hundred classmates that drew better, folded better, and even colored in the lines better. But as a child raised in a household that couldn't afford Hot Wheels and Silly Bandz, I was forced to quench my boredom with index cards and the blank sides of my father's English notes. I still recall my prized possession: a glistening mechanical pencil with a detachable cap, smuggled from my father's backpack in a midnight heist. What followed was endless hours of scribbling, cutting, and crafting. My worksheets rarely lacked stickmen climbing up the sides, and some amalgamation of monstrous creatures could always be found on the back.
Ironic as it is, these flimsy sheets of paper formed the solid backbone of my childhood, and it could be found everywhere; a Calvin and Hobbes cartoon from the Sunday paper could be spotted jammed between two untouched novels. On the floor lay an uncleaned chess set, consisting of a board drawn in with a pencil, and pieces made from fractions of index cards. Even the cardboard Budweiser box, which we called our dining table for several months, had first been made from paper. A photograph of my brother's birthday still sticks with me. My shabbily-cut paper crowns decorate our heads, as we all pose blowing out the candles of a Star Market cake. If you ignore the bed-less rooms, stained walls, and scratched-up linoleum, we are a normal family, celebrating a normal birthday. I could use some verbose synonym, but no word quite captures this emotion in our eyes than "happy". We were happy.
My meaningless doodles slowly evolved into full comic strips and playing cards, which acted as a source of entertainment for me and my little brother. We mimicked Nintendo games with nothing but drawings and a pinch of imagination, occasionally employing dice for an exciting aspect of randomness. Before every road trip, I recall packing a stack of paper and two pencils for us to share, and we would take turns arguing which of our fantastic creatures would win in single combat. As I reminisce on my childhood projects, I now realize that paper brought us together in a way nothing ever has. But it wasn't just paper alone; it was our individual ideas and unique imaginations that allowed us to bond over a common interest. Paper acted as an invitation for us to visually express our thoughts, but it stood purposeless without an artist at the helm.
A simple dictionary entry cannot illustrate a child's bliss as they bring an imaginary friend to life within cartoons, nor can it define the connection between brothers as they discuss the optimal method of folding paper airplanes. The internet will define paper as a sheet to write or draw on, failing to capture its extraordinary ability to mold into whatever the artist desires. I view the world similarly–– not as a rigid canvas but as an empty sheet, taunting us to dump our unique ideas to leave a signature on the face of history. When life folds at unexpected times, it is our job to learn from the creases and explore new paths to reach our final product. And when the page is filled to the brim with brilliant thoughts, a blank slate awaits on the other side.
"""
outputs = clf_pipeline(new_essay)
threshold = 0.5
predicted_labels = [label_names[i] for i, score in enumerate(outputs[0]) if score['score'] > threshold]
print(predicted_labels)

Device set to use cuda:0


['Expand Initial Failure Scene', 'Needs Clearer Timeline', 'Needs Conflict Arc', 'Needs More Self-focus', 'Smoother Transition to Broader Reflection', 'Strengthen Closing Reflection', 'Too Abstract', 'Weaker Reflection Arc']
